In [1]:
# https://www.youtube.com/watch?v=VtRLrQ3Ev-U

# https://keras.io/examples/nlp/text_classification_from_scratch/

In [2]:
# Install necessary packages
!pip install numpy pandas tensorflow scikit-learn seaborn matplotlib

In [3]:
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import train_test_split
# chatgpt to help with accuracy (add weighting)
from sklearn.utils.class_weight import compute_class_weight 
# visualise confusion matrix
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
# chatgpt to save the mappings (vectorizer and labels)
import pickle


Load and look through data.

In [4]:
df = pd.read_csv("labeledData3.csv")

In [5]:
print(df["Food"])

0               asparagus
1                avocados
2                 bananas
3       wholewheat bagels
4              white buns
              ...        
1027          cotton pads
1028            batteries
1029         sponges pack
1030        rubber gloves
1031      food containers
Name: Food, Length: 1032, dtype: object


In [6]:
print(df["Categories"])

0                                     produce
1                                     produce
2                                     produce
3                             breads & grains
4                             breads & grains
                        ...                  
1027    household supplies and non-food items
1028    household supplies and non-food items
1029    household supplies and non-food items
1030    household supplies and non-food items
1031    household supplies and non-food items
Name: Categories, Length: 1032, dtype: object


Converting the categories column into numeric labels (label encoding).

In [7]:
df["label"] = df["Categories"].astype("category").cat.codes
labels = df["label"].to_numpy()
category_names = list(df["Categories"].astype("category").cat.categories)
num_classes = df["label"].nunique()

print(labels)
print(category_names)
print(num_classes)

[8 8 8 ... 4 4 4]
['beverages', 'breads & grains', 'dairy & eggs', 'frozen products', 'household supplies and non-food items', 'meat & seafood', 'oils, condiments & sauces', 'pantry & dry goods', 'produce', 'ready & instant meals', 'snacks & sweets']
11


Converting food column into a list.

In [8]:
texts=df["Food"].tolist()

print(texts)

['asparagus', 'avocados', 'bananas', 'wholewheat bagels', 'white buns', 'brown buns', 'sourdough', 'rye bread', 'wholemeal bread', 'tortillas', 'bakery rolls', 'fresh bread', 'rice noodles', 'somen noodles', 'ripe berries', 'red berries', 'organic vegetables', 'brocolli', 'ripe apples', 'salad leaves', 'mixed peppers', 'brown rice', 'cauliflower- 1kg', 'american cheese', 'coffee', 'corn on the cob', 'dark chocolate', 'grapes', 'milk', 'stinky bishop cheese', 'mature cheddar', 'shredded cheese', 'mozzarella block', 'milk 2l', 'yogurt pots', 'free-range eggs', 'jersey gold-top milks', 'mango juice', 'juices', 'water', 'oolong teas', 'iced latte', 'hot chocolate', 'flat white coffee', 'cappacino coffee', 'mocha coffee', 'soft drinks', 'fizzy drinks', 'energy drinks', 'cranberry juice', 'cashew nuts', 'prawn cocktail crisps', 'salt and vinegar crisps', 'biscuit bars', 'cookies box', 'candy', 'snack packs', 'treat size', 'salted peanuts', 'chocolate coated pretzels', 'toffee popcorn', 'swee

Split into test and training sets.

In [9]:
X_train, X_test, y_train, y_test = train_test_split(
    texts, labels, test_size=0.2, random_state=42
)

[Vectorize](https://keras.io/api/layers/preprocessing_layers/text/text_vectorization/) the text. 

In [10]:
# chatGPT helped adding max_tokens value and output_sequence_length value (suggested: 5000 and 12 for my dataset size, but I tried a few different combos)
vectorized_text = tf.keras.layers.TextVectorization(
    max_tokens = 10000,
    output_mode = "int",
    output_sequence_length = 20,
)

vectorized_text.adapt(texts)
print(vectorized_text)

vocab_size = len(vectorized_text.get_vocabulary())
print(vocab_size)

<TextVectorization name=text_vectorization, built=False>


In [11]:
# chatgpt suggestion to help with accuracy (add weighting)
weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(y_train),
    y=y_train
)

class_weights = dict(enumerate(weights))
print(class_weights)

{0: np.float64(1.4150943396226414), 1: np.float64(0.7425742574257426), 2: np.float64(1.3888888888888888), 3: np.float64(1.875), 4: np.float64(1.4423076923076923), 5: np.float64(1.5957446808510638), 6: np.float64(1.9736842105263157), 7: np.float64(1.530612244897959), 8: np.float64(0.4934210526315789), 9: np.float64(0.3787878787878788), 10: np.float64(1.829268292682927)}


Build model.

In [12]:
model = tf.keras.Sequential([
    vectorized_text,
    tf.keras.layers.Embedding(vocab_size, 128),
    tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(32)),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dense(num_classes, activation='softmax')
])


NameError: name 'vocab_size' is not defined

In [ ]:
model.compile(optimizer="adam",
              loss="sparse_categorical_crossentropy",
              metrics=["accuracy"])

In [ ]:
# ChatGPT for error ValueError: Unrecognized data type: x=['vanilla yogurt',
# tf.constant allows you to create a tensor from tensor like objects like lists (https://www.geeksforgeeks.org/python/python-tensorflow-constant/)
X_train = tf.constant(X_train)
X_test  = tf.constant(X_test)

print(X_train)
print(X_test)

tf.Tensor(
[b'argan oil' b'acai' b'sloppy joe' b'raisins' b'lemon pudding'
 b'cappelletti' b'chilli flakes' b'mac and cheese' b'grape seed oil'
 b'spirelli' b'black rice' b'yorkshire pudding' b'peanut butter'
 b'butter paneer meal' b'kebab meat' b'washing up liquid'
 b'margarine spread' b'butter' b'quattro formaggi pizza' b'oxtail soup'
 b'chocolate frozen yogurt' b'strawberry guava' b'turkey' b'skimmed milk'
 b'strawberry ice cream' b'mushroom pizza' b'sparkling mineral water'
 b'potato soup' b'eggnog' b'isotonic drink' b'sambal olek'
 b'stinky bishop cheese' b'vegetable oil spread fat free'
 b'tapioca pudding' b'cranberry juice' b'dishes' b'calabrese pizza'
 b'beetroot' b'tuna fish' b'tomato soup' b'chappati' b'whipped/sour cream'
 b'self raising flour' b'duck spring rolls' b'oyster raw' b'green olives'
 b'deodorant' b'fish and chips' b'sweet potato' b'whiskey sour mix'
 b'frankfurter' b'hazelnut oil' b'strawberry icecream' b'raspberries'
 b'vegetable pizza' b'apricot kernel oil' b'v

Train Model.

In [ ]:
# tried different epochs to improve accuracy
model.fit(
    X_train, y_train,
    epochs=10,
    class_weight=class_weights
)

Epoch 1/10
26/26 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.0485 - loss: 2.4015
Epoch 2/10
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.2788 - loss: 2.3768
Epoch 3/10
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.4218 - loss: 2.3109
Epoch 4/10
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.4461 - loss: 2.0467
Epoch 5/10
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.6400 - loss: 1.5658
Epoch 6/10
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.7806 - loss: 1.1134
Epoch 7/10
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.9200 - loss: 0.6809
Epoch 8/10
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.9430 - loss: 0.4026
Epoch 9/10
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.9624 - loss: 0.2792
Epoch 10/10
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.9758 - loss: 0.1664


Evaluate (using a confusion matrix and model.evaluate) and make changes in sataset if necessary.

In [ ]:
model.evaluate(X_test, y_test)

7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.5411 - loss: 1.4848  


[1.4847772121429443, 0.5410627722740173]

In [ ]:
model.evaluate(X_train, y_train)

26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9782 - loss: 0.1501


[0.15009960532188416, 0.9781818389892578]

In [ ]:
# accuracy super low test accuracy: 0.1212 - loss: 2.2419 train accuracy: 0.1769 - loss: 2.1160
# increased epochs, max tokens and output sequence length still super low
# manually label more for the dataset
# maybe manually label entire dataset (soo 500+ for training), then make a new dataset to be classified
# using foodTrainingSubset csv accuracy increase test==-.4809, train==.5072
# re-cleaning with trainingdata-cleaning notebook test==.4605, train==.5099

In [ ]:
y_pred_probs = model.predict(X_test)
y_pred = np.argmax(y_pred_probs, axis=1)

cm = confusion_matrix(y_test, y_pred)
print(cm)

7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step
[[ 6  0  0  0  0  0  0  0  2  1  3]
 [ 0 10  1  0  0  0  1  2  0  3  6]
 [ 0  1  7  2  0  0  0  0  1  3  1]
 [ 0  0  0  6  0  1  0  0  0  0  1]
 [ 0  0  0  0  7  0  0  0  3  0  4]
 [ 0  0  0  0  0 13  0  0  0  0  1]
 [ 0  0  0  0  0  0  6  1  1  0  5]
 [ 0  0  0  0  0  0  2  9  0  1  2]
 [ 2  0  0  0  0  1  2  1 10  1 21]
 [ 0  0  1  0  0  4  0  0  2 30  6]
 [ 0  0  1  0  0  0  1  2  1  0  8]]


ChatGPT helped on how to save this model to be used for all the tasks.

In [4]:
model.save("grocery_classifier.keras")

# Save vectorizer
with open("vectorizer.pkl", "wb") as f:
    pickle.dump(vectorized_text, f)

# Save label map
with open("label_encoder.pkl", "wb") as f:
    pickle.dump(category_names, f)

NameError: name 'model' is not defined

Load Model and CLI.

In [5]:
model = tf.keras.models.load_model("grocery_classifier.keras")

with open("vectorizer.pkl", "rb") as f:
    vectorizer = pickle.load(f)

with open("label_encoder.pkl", "rb") as f:
    label_map = pickle.load(f)  

In [6]:
def predict_category(item_name: str) -> str:
    input_tensor = tf.constant([item_name])

    pred_probs = model.predict(input_tensor)
    label_index = int(tf.argmax(pred_probs, axis=1)[0])
    category = label_map[label_index]
    return category

In [7]:
if __name__ == "__main__":
    print("Type a grocery item name or 'quit' to exit.")

    while True:
        item = input("Enter item: ").strip()
        if item.lower() in ["quit", "exit"]:
            break

        category = predict_category(item)
        print(f"Category: {category}")

# some finally work! still high inaccuracy but frozen products and (some) breads and grains are accurate.

Type a grocery item name or 'quit' to exit.

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 212ms/step
 → Category: beverages

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step
 → Category: snacks & sweets

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step
 → Category: pantry & dry goods

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step
 → Category: breads & grains

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step
 → Category: breads & grains

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step
 → Category: frozen products

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step
 → Category: frozen products

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step
 → Category: breads & grains

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step
 → Category: snacks & sweets

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step
 → Category: snacks & sweets

